# Bonus Slideshow Vidéo - Générateur de Slideshow (Optionnel)

**Navigation** : [<< Précédent (01-1)](01-1-Video-Operations-Basics.ipynb) | [Index](../../README.md) | [Suivant >> (01-2)](01-2-GPT-5-Video-Understanding.ipynb)

**Module :** 01-Video-Foundation  
**Niveau :** Debutant  
**Type :** Accrétion bonus (optionnelle)  
**Duree estimee :** 15 minutes  
**VRAM :** 0 (CPU uniquement)

> **Ce notebook est une accrétion optionnelle**, extraite de la conclusion de
> [01-1-Video-Opérations-Basics](01-1-Video-Operations-Basics.ipynb) pour rendre la
> lane principale plus claire. Il reprend intégralement le challenge qui figurait en
> fin du 01-1, sans modification du fond ni de la forme. Les étudiants qui veulent
> transformer les manipulations moviepy/imageio du 01-1 en **produit livrable**
> trouvent ici le terrain de jeu ; les autres passent directement au 01-2.

## Contexte pédagogique

Le 01-1 a posé les fondamentaux :

- **Section 1** : création d'une vidéo de test à partir de frames générées avec PIL
  (`Image.new`, `ImageDraw`) et assemblage avec `imageio.get_writer().append_data()`.
- **Sections 2-4** : manipulations moviepy (trim, concaténation, overlay texte),
  sonde ffmpeg-python, extraction rapide de frames avec decord.

Le challenge ci-dessous remet ces primitives au service d'un **livrable fini** : un
slideshow 5 secondes @ 1 FPS avec cinq messages sur cinq fonds colorés différents.
C'est la transition entre « savoir manipuler une vidéo » et « savoir produire un
objet vidéo autonome ».

## Prérequis

- Avoir complété (ou parcouru) le notebook 01-1.
- Packages : `Pillow`, `imageio`, `imageio-ffmpeg` (déjà listés dans
  `requirements-video.txt` à la racine de `GenAI/`).
- FFmpeg installé sur le système (`winget install FFmpeg` ou `apt install ffmpeg`).


***

## CHALLENGE BONUS - Générateur de Slideshow Vidéo

**Points : 0.5 pts**

## Objectif

Créer une vidéo **slideshow** à partir de frames générées programmatiquement.

## Ce que vous avez appris

La Section 1 montre:
- Génération de frames avec PIL (`Image.new`, `ImageDraw`)
- Assemblage avec imageio (`get_writer`, `append_data`)

## Critères de succes

- [ ] Générer **5 frames** affichant un message différent chacune:
  - Frame 1: "Bienvenue"
  - Frame 2: "Voici le sujet"
  - Frame 3: "Point important"
  - Frame 4: "Conclusion"
  - Frame 5: "Merci"
- [ ] Chaque frame doit avoir:
  - Une couleur de fond différente (gradient ou palette cohérente)
  - Le texte centré
- [ ] Assembler en vidéo **5 secondes @ 1 FPS**
- [ ] Sauvegarder dans `OUTPUT_DIR`

## Contraintes techniques

- Résolution: 640x480
- Utiliser PIL pour les frames, imageio pour l'assemblage
- FPS = 1 signifie chaque frame dure 1 seconde

***

**Soumission** : PR avec titre "Challenge #7 - [Votre Nom]", aperçu des frames et vidéo générée


## Espace de travail du challenge

Le challenge ci-dessus était resté **sans endroit où répondre** : ce notebook est l'espace de travail. Deux **exemples guidés** rappellent les primitives posées dans [01-1](01-1-Video-Operations-Basics.ipynb) (une frame PIL centrée, un mini assemblage imageio). Trois autres exemples guidés — **crédités au Groupe G15 (PR #21040)**, rendu de TP — portent le livrable complet du challenge : les exercices 1 à 3 qu'ils résolvent deviennent des exemples, et trois **exercices neufs** (4 à 6) reprennent la même mécanique sur des variantes (fonds en dégradé, carte de titre en préambule, vérification au pixel près). Tout tourne sur CPU (VRAM 0, conformément à l'en-tête) : `Pillow` pour les frames, `imageio` + `imageio-ffmpeg` (binaire ffmpeg embarqué) pour l'assemblage. Les artefacts sont écrits dans `outputs/video_01_1b_slideshow/` (non versionné).


### Exemple guidé 1 — Une frame PIL : fond coloré, message centré

La brique de base du slideshow. La fonction `frame_demo` crée une image RGB 640x480, remplit le fond, puis centre le texte en calculant sa boîte englobante.

Le centrage se calcule, il ne se devine pas : on mesure d'abord la boîte du texte (`draw.textbbox`), on obtient sa largeur et sa hauteur réelles pour la police choisie, puis on place le coin en `((largeur_frame - largeur_texte) / 2, (hauteur_frame - hauteur_texte) / 2)`. Toute autre approche — une marge fixe, un nombre de caractères — se décale dès que la police ou le message change.


In [1]:
from pathlib import Path
from PIL import Image, ImageDraw

OUTPUT_DIR = Path("outputs") / "video_01_1b_slideshow"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def frame_demo(message, couleur, taille=(640, 480), chemin=None):
    """Genere une frame PIL : fond colore uniforme, message centre, texte blanc."""
    img = Image.new("RGB", taille, couleur)
    dessin = ImageDraw.Draw(img)
    bbox = dessin.textbbox((0, 0), message)
    x = (taille[0] - (bbox[2] - bbox[0])) // 2
    y = (taille[1] - (bbox[3] - bbox[1])) // 2
    dessin.text((x, y), message, fill="white")
    if chemin is not None:
        img.save(chemin)
    return img


frame_exemple = frame_demo("Demo A", (40, 60, 120), chemin=OUTPUT_DIR / "frame_demo.png")
print(f"Frame exemple sauvegardee : {OUTPUT_DIR / 'frame_demo.png'} ({frame_exemple.size})")


Frame exemple sauvegardee : outputs\video_01_1b_slideshow\frame_demo.png ((640, 480))


### Exemple guidé 2 — Mini assemblage : 2 frames @ 1 FPS

Le montage minimal : à 1 FPS, chaque frame dure une seconde. Deux frames donnent donc une vidéo de 2 secondes — le livrable du challenge est la même mécanique répétée 5 fois.

**Ce que `fps` signifie exactement ici.** À 1 image par seconde, chaque frame dure une seconde, donc N frames donnent N secondes : la durée d'un slideshow est un choix de `fps`, pas une propriété des images. Les mêmes cinq frames écrites à 24 fps dureraient 0,21 s ; et le writer exige des frames de même forme, ce qui impose de fixer la taille une fois pour toutes.


In [2]:
import numpy as np
import imageio.v2 as imageio

frames_exemple = [
    frame_demo("Demo A", (40, 60, 120)),
    frame_demo("Demo B", (120, 60, 40)),
]
video_exemple = OUTPUT_DIR / "mini_slideshow_exemple.mp4"
with imageio.get_writer(video_exemple, fps=1) as writer:
    for f in frames_exemple:
        writer.append_data(np.asarray(f))
print(f"Video exemple : {video_exemple.name} ({video_exemple.stat().st_size} octets, "
      f"{len(frames_exemple)} frames @ 1 FPS = {len(frames_exemple)} s)")


Video exemple : mini_slideshow_exemple.mp4 (2340 octets, 2 frames @ 1 FPS = 2 s)


### Exemple guidé 3 — Les 5 frames du challenge *(crédité : Groupe G15, PR #21040)*

Produire la liste des 5 frames imposées par les critères de succès : une par message, chacune sur un fond **distinct** (palette cohérente de votre choix), texte centré, résolution 640x480.

La solution créditée ci-dessous est conservée telle que rendue. Elle montre deux raffinements par rapport à l'exemple guidé 1 : le centrage utilise la **boîte englobante réelle** du texte (en compensant son décalage vertical, surtout perceptible avec une vraie police), et une **garde d'entrée** refuse un message vide — un livrable se protège aussi de ses entrées.


In [3]:
# Exemple guide 3, solution creditée (Groupe G15, PR #21040)
# Ecart declare : robustesse du chargement de police ajoute (sur machine sans
# DejaVuSans systeme, le fallback initial produisait la police bitmap par defaut) ;
# tout le reste est conserve tel que rendu.
from PIL import ImageFont

MESSAGES = ["Bienvenue", "Voici le sujet", "Point important", "Conclusion", "Merci"]
PALETTE = [(75, 22, 55), (105, 30, 65), (135, 38, 75), (165, 46, 85), (195, 54, 95)]

def creer_frame(message, couleur, taille=(640, 480)):
    """Centre la boite reelle du texte, y compris son decalage vertical."""
    if not message.strip():
        raise ValueError("Le message doit contenir du texte")
    image = Image.new("RGB", taille, couleur)
    dessin = ImageDraw.Draw(image)
    try:
        police = ImageFont.truetype("DejaVuSans.ttf", 36)
    except OSError:
        try:
            import matplotlib
            police = ImageFont.truetype(
                Path(matplotlib.get_data_path()) / "fonts" / "ttf" / "DejaVuSans.ttf", 36)
        except OSError:
            police = ImageFont.load_default()
    gauche, haut, droite, bas = dessin.textbbox((0, 0), message, font=police)
    position = ((taille[0] - (droite-gauche))/2 - gauche,
                (taille[1] - (bas-haut))/2 - haut)
    dessin.text(position, message, font=police, fill="white")
    return image

frames_challenge = [creer_frame(m, c) for m, c in zip(MESSAGES, PALETTE)]
assert len(frames_challenge) == 5 and len(set(PALETTE)) == 5
assert all(f.size == (640, 480) and f.mode == "RGB" for f in frames_challenge)
for i, frame in enumerate(frames_challenge, 1):
    frame.save(OUTPUT_DIR / f"frame_{i}.png")
try:
    creer_frame("", PALETTE[0])
except ValueError:
    print("Cas limite : message vide refuse")
else:
    raise AssertionError("Un message vide devrait etre refuse")
print("5 images 640x480 generees, couleurs distinctes et texte centre")


Cas limite : message vide refuse
5 images 640x480 generees, couleurs distinctes et texte centre


## Exercice 4 — Variante : cinq frames en dégradé vertical

Même livrable que l'exemple guidé 3, mais les fonds ne sont plus des aplats : chaque frame porte un **dégradé vertical** entre deux couleurs de votre palette — la première couleur en haut de frame, la seconde en bas. Les cinq frames restent en 640x480, une par message de `MESSAGES`, texte centré.

- `creer_frame` ne réutilise pas telle quelle (elle remplit un aplat) : écrire `frame_degrade(message, haut, bas, taille=(640, 480))` qui construit le fond ligne par ligne puis centre le texte comme l'exemple guidé 3.
- **Indice** : le fond se construit en une passe avec `np.linspace(0, 1, taille[1])` broadcasté sur les trois canaux, puis `Image.fromarray`.
- **Etape 1** : interpoler chaque canal entre `haut` et `bas`. **Etape 2** : convertir en image PIL. **Etape 3** : centrer le texte (boîte englobante réelle).


In [4]:
# Exercice 4 (a completer) : cinq frames en degrade vertical
# TODO Etudiant : definir frame_degrade(message, haut, bas, taille=(640, 480))
# puis construire frames_defile = liste de 5 PIL Images 640x480, une par message
# de MESSAGES, chacune sur un degrade vertical (couleur haut -> couleur bas).
frames_defile = None  # TODO etudiant : liste de 5 PIL.Image en degrade
print("Exercice 4 a completer :",
      None if frames_defile is None else f"{len(frames_defile)} frames en degrade")


Exercice 4 a completer : None


### Exemple guidé 4 — L'assemblage : 5 secondes @ 1 FPS *(crédité : Groupe G15, PR #21040)*

Assembler `frames_challenge` en vidéo **slideshow 5 s @ 1 FPS** (5 frames, une seconde chacune), et sauvegarder dans `OUTPUT_DIR` sous le nom `slideshow_5s.mp4`.

La solution créditée reprend le mini-assemblage de l'exemple guidé 2 en ajoutant les deux options de production qui rendent le fichier lisible partout : `codec="libx264"` et `pixelformat="yuv420p"`.


In [5]:
# Exemple guide 4, solution creditée telle que rendue (Groupe G15, PR #21040)
video_challenge = OUTPUT_DIR / "slideshow_5s.mp4"
with imageio.get_writer(video_challenge, fps=1, codec="libx264", pixelformat="yuv420p") as writer:
    for frame in frames_challenge:
        writer.append_data(np.asarray(frame))
print("Video creee :", video_challenge, video_challenge.stat().st_size, "octets")


Video creee : outputs\video_01_1b_slideshow\slideshow_5s.mp4 13797 octets


## Exercice 5 — Variante : carte de titre en préambule

Le livrable gagne une carte de titre : produire `slideshow_titre.mp4`, **6 frames @ 1 FPS** (6 secondes) — une frame de titre générée avec `frame_demo` (message de votre choix, couleur neutre distincte des cinq), suivie des cinq frames en dégradé de l'exercice 4.

- Écrire la séquence dans `OUTPUT_DIR / "slideshow_titre.mp4"` avec les mêmes options de codec que l'exemple guidé 4 (`fps=1`, `codec="libx264"`, `pixelformat="yuv420p"`).
- **Indice** : la carte de titre est une `PIL.Image` comme les autres — la séquence finale est une simple concaténation de listes.


In [6]:
# Exercice 5 (a completer) : carte de titre en preambule
# TODO Etudiant : construire slideshow_titre.mp4 = 6 frames @ 1 FPS :
# 1 frame de titre (frame_demo) + les 5 frames_defile de l'exercice 4,
# sauvegardees dans OUTPUT_DIR comme l'exemple guide 4.
video_titre = None  # TODO etudiant : Path du fichier slideshow_titre.mp4 produit
print("Exercice 5 a completer :", video_titre)


Exercice 5 a completer : None


### Exemple guidé 5 — Vérifier le livrable *(crédité : Groupe G15, PR #21040)*

Un livrable se vérifie avant de se soumettre. La solution créditée en fait une **fonction réutilisable** : elle refuse un fichier absent ou vide, relit les métadonnées (`fps`, `duration`, `size`) **et** compte les frames, et n'accepte le verdict `ok` que si les attendus sont tous réunis — 5 frames, 5,0 s, 1 FPS, 640x480. Indices : `imageio.get_reader(...)` expose `count_frames()` et `get_meta_data()`.


In [7]:
# Exemple guide 5, solution creditée telle que rendue (Groupe G15, PR #21040)
def verifier_video(chemin):
    verdict = {"existe": chemin.is_file(), "nframes": None, "duree_s": None, "ok": False}
    if not verdict["existe"] or chemin.stat().st_size == 0:
        return verdict
    try:
        with imageio.get_reader(chemin) as lecteur:
            meta = lecteur.get_meta_data()
            verdict["nframes"] = lecteur.count_frames()
            verdict["duree_s"] = meta.get("duration")
            verdict["ok"] = (verdict["nframes"] == 5
                and verdict["duree_s"] is not None
                and abs(verdict["duree_s"] - 5.0) < 0.05
                and abs(meta.get("fps", 0) - 1.0) < 0.001
                and tuple(meta.get("size", ())) == (640, 480))
    except (OSError, ValueError, RuntimeError) as erreur:
        verdict["erreur"] = str(erreur)
    return verdict

verdict_livrable = verifier_video(video_challenge)
assert verdict_livrable["ok"], verdict_livrable
assert not verifier_video(OUTPUT_DIR / "absent_test.mp4")["ok"]
print(verdict_livrable)


{'existe': True, 'nframes': 5, 'duree_s': 5.0, 'ok': True}


## Exercice 6 — Variante : vérification au pixel près

Les métadonnées ne disent pas ce que **montre** la vidéo. Étendre la vérification de l'exemple guidé 5 à `slideshow_titre.mp4` : relire les frames une à une (`imageio.get_reader(...)` puis `get_data(i)`), et construire `verdict_pixels = {"nframes": int|None, "coin_haut_gauche_titre": tuple|None, "coin_bas_droit_dernier": tuple|None, "ok": bool}` où `ok` exige 6 frames, le coin haut-gauche de la première frame proche de la couleur de la carte de titre, et le coin bas-droit de la dernière frame proche de la couleur basse du dernier dégradé.

- **Indice 1** : `reader.get_data(0)` donne la première frame ; son pixel `[0, 0]` est le coin haut-gauche.
- **Indice 2** : comparer à tolérance près (écart absolu ≤ 10 par canal, par exemple), jamais à l'égalité stricte — les encodages vidéo perturbent légèrement les valeurs.
- **Etape 1** : compter les frames. **Etape 2** : extraire les deux coins. **Etape 3** : comparer aux couleurs attendues et conclure.


In [8]:
# Exercice 6 (a completer) : verification au pixel pres
# TODO Etudiant : construire verdict_pixels pour slideshow_titre.mp4 :
# relire les 6 frames, extraire le coin haut-gauche de la frame 1 et le coin
# bas-droit de la frame 6, et valider ok = 6 frames ET coins coherents aux
# couleurs attendues (tolerance par canal).
verdict_pixels = None  # TODO etudiant
print("Exercice 6 a completer :", verdict_pixels)


Exercice 6 a completer : None


## Conclusion

Les trois exemples crédités s'enchaînent en une chaîne de production complète : des frames aux fonds distincts et au texte réellement centré, un assemblage 5 frames @ 1 FPS que la vérification relit depuis le fichier produit (`nframes` = 5, `duree_s` = 5.0, verdict `ok`), et un garde-fou qui refuse tout aussi bien un fichier absent qu'un message vide. La solution de vérification teste aussi le cas d'erreur — c'est ce qui distingue une démo d'un livrable. Les exercices 4 à 6 reprennent exactement ces primitives en changeant un seul paramètre à la fois : la nature du fond (dégradé), la longueur de la séquence (carte de titre), la profondeur de la vérification (pixel plutôt que métadonnée).
